# Hugging Face Tabular 모델 벤치마킹

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. 환경 설정

먼저, 각 모델에 필요한 라이브러리를 설치합니다. 여기에는 TabPFN을 위한 `tabpfn`, AutoGluon을 위한 `autogluon.tabular[all]`, 일반적인 Hugging Face 모델 상호작용을 위한 `transformers`, 그리고 데이터 처리 및 평가를 위한 기타 유틸리티가 포함됩니다.

In [27]:
!pip install tabpfn autogluon.tabular[all] transformers datasets scikit-learn pandas -q

In [28]:
from huggingface_hub import hf_hub_download
print("huggingface hub ok")

huggingface hub ok


In [36]:
import subprocess

# 1) tabpfn_v2: model_path="Prior-Labs/TabPFN-v2-reg"는 tabpfn 라이브러리가 인식하는
#    체크포인트 이름이 아니라서 다운로드가 실패한다. 기본 체크포인트를 쓰도록 고친다.
path = "/content/drive/MyDrive/Colab Notebooks/benchmark.py"
src = open(path, encoding="utf-8").read()

old = (
    "            return TabPFNRegressor(\n"
    "                model_path=\"Prior-Labs/TabPFN-v2-reg\",\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
new = (
    "            return TabPFNRegressor(\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
count = src.count(old)
assert count == 1, f"expected 1 match, got {count}"
src = src.replace(old, new)
open(path, "w", encoding="utf-8").write(src)
print("tabpfn_v2 model_path 버그 수정 완료")

# 2) nori: synthefy_nori 패키지가 설치돼 있는지 / pip에 실존하는지 확인
result = subprocess.run(
    ["pip", "install", "synthefy-nori", "-q"],
    capture_output=True, text=True,
)
print("\n--- pip install synthefy-nori ---")
print("returncode:", result.returncode)
print(result.stdout[-1500:])
print(result.stderr[-1500:])

tabpfn_v2 model_path 버그 수정 완료

--- pip install synthefy-nori ---
returncode: 0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.3/285.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 27.7 MB/s eta 0:00:00

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.6 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.25.1 which is incompatible.



In [38]:
path = "/content/drive/MyDrive/Colab Notebooks/benchmark.py"
src = open(path, encoding="utf-8").read()

old = (
    "            return TabPFNRegressor(\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
new = (
    "            return TabPFNRegressor(\n"
    "                model_path=\"tabpfn-v2-regressor-v2_default.ckpt\",\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
count = src.count(old)
assert count == 1, f"expected 1 match, got {count}"
src = src.replace(old, new)
open(path, "w", encoding="utf-8").write(src)
print("model_path를 Prior-Labs/TabPFN-v2-reg 내 체크포인트 파일명으로 명시 완료")

model_path를 Prior-Labs/TabPFN-v2-reg 내 체크포인트 파일명으로 명시 완료


## 2. 모델 설명 및 실행

이제 각 모델을 로드하고 기본적인 예측을 시연합니다. 벤치마킹을 위해서는 더미 데이터를 실제 데이터셋으로 교체해야 합니다.

### 2-1. Prior-Labs/TabPFN-v2-reg

이 모델은 회귀 작업을 위한 Tabular 사전 학습형 신경망입니다. `tabpfn` 라이브러리를 활용합니다.

In [40]:
path = "/content/drive/MyDrive/Colab Notebooks/benchmark.py"
src = open(path, encoding="utf-8").read()

old = (
    "        if self.name == \"tabpfn_v2\":\n"
    "            try:\n"
    "                from tabpfn import TabPFNRegressor\n"
    "            except ImportError as exc:\n"
    "                raise RuntimeError(\n"
    "                    \"TabPFN 실행에는 `pip install tabpfn`이 필요합니다.\"\n"
    "                ) from exc\n"
    "            return TabPFNRegressor(\n"
    "                model_path=\"tabpfn-v2-regressor-v2_default.ckpt\",\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
new = (
    "        if self.name == \"tabpfn_v2\":\n"
    "            try:\n"
    "                from tabpfn import TabPFNRegressor\n"
    "            except ImportError as exc:\n"
    "                raise RuntimeError(\n"
    "                    \"TabPFN 실행에는 `pip install tabpfn`이 필요합니다.\"\n"
    "                ) from exc\n"
    "            from huggingface_hub import hf_hub_download\n"
    "            local_ckpt = hf_hub_download(\n"
    "                repo_id=\"Prior-Labs/TabPFN-v2-reg\",\n"
    "                filename=\"tabpfn-v2-regressor-v2_default.ckpt\",\n"
    "            )\n"
    "            return TabPFNRegressor(\n"
    "                model_path=local_ckpt,\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
count = src.count(old)
assert count == 1, f"expected 1 match, got {count}"
src = src.replace(old, new)
open(path, "w", encoding="utf-8").write(src)
print("tabpfn_v2: HF에서 직접 로컬로 받아서 로딩하도록 수정 완료")

tabpfn_v2: HF에서 직접 로컬로 받아서 로딩하도록 수정 완료


In [41]:
import json
from pathlib import Path
import pandas as pd

MODEL = "tabpfn_v2"
artifact_dir = Path("/content/drive/MyDrive/Colab Notebooks") / f"colab_{MODEL}"

!python "/content/drive/MyDrive/Colab Notebooks/benchmark.py" benchmark \
    --data "/content/drive/MyDrive/Colab Notebooks/sizekorea_measurements_clean.csv" \
    --models {MODEL} \
    --artifact-dir "{artifact_dir}" \
    --height 170 \
    --weight 65

metrics_path = artifact_dir / "metrics.json"
if metrics_path.exists():
    df = pd.DataFrame(json.loads(metrics_path.read_text(encoding="utf-8")))
    display(df)
else:
    print(f"{MODEL}: metrics.json이 생성되지 않았습니다 — 위 실행 로그의 에러를 확인하세요.")


tabpfn-v2-regressor-v2_default.ckpt: downloading bytes:  61% 27.3M/44.4M [00:00<00:00, 40.8MB/s, 1.77MB/s  ]
tabpfn-v2-regressor-v2_default.ckpt: downloading bytes: 100% 41.3M/41.3M [00:01<00:00, 38.0MB/s, 3.98MB/s  ]
tabpfn-v2-regressor-v2_default.ckpt: reconstructing file: 100% 44.4M/44.4M [00:01<00:00, 40.9MB/s, 4.33MB/s  ]
/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as a different type depending on the ColumnTransformer inputs.
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/compose/_c

,model,target,mae,rmse,r2,p90_absolute_error,fit_seconds,predict_ms_per_row
0,tabpfn_v2,chest,2.484054,3.161852,0.867479,5.002925,8.208138,9.849576
1,tabpfn_v2,waist,3.317963,4.136839,0.833903,6.463167,8.208138,9.849576
2,tabpfn_v2,hip,2.382576,3.056910,0.746575,5.034064,8.208138,9.849576
3,tabpfn_v2,thigh,2.261006,2.905415,0.651467,4.764128,8.208138,9.849576
4,tabpfn_v2,calf,1.112673,1.443447,0.778268,2.376257,8.208138,9.849576
5,tabpfn_v2,arm,1.086886,1.438839,0.818118,2.159990,8.208138,9.849576
6,tabpfn_v2,shoulder,1.151851,1.458336,0.731912,2.410880,8.208138,9.849576


### 2-2. Synthefy/Nori

Nori는 트랜스포머 기반의 테이블형 모델입니다. `benchmark.py`는 `synthefy_nori` 패키지의 `NoriRegressor`를 사용합니다.

In [42]:
import json
from pathlib import Path
import pandas as pd

MODEL = "nori"
artifact_dir = Path("/content/drive/MyDrive/Colab Notebooks") / f"colab_{MODEL}"

!python "/content/drive/MyDrive/Colab Notebooks/benchmark.py" benchmark \
    --data "/content/drive/MyDrive/Colab Notebooks/sizekorea_measurements_clean.csv" \
    --models {MODEL} \
    --artifact-dir "{artifact_dir}" \
    --height 170 \
    --weight 65

metrics_path = artifact_dir / "metrics.json"
if metrics_path.exists():
    df = pd.DataFrame(json.loads(metrics_path.read_text(encoding="utf-8")))
    display(df)
else:
    print(f"{MODEL}: metrics.json이 생성되지 않았습니다 — 위 실행 로그의 에러를 확인하세요.")

model  mean_mae  mean_rmse  mean_r2  mean_p90_error  fit_seconds  predict_ms_per_row
 nori  1.979971   2.527567 0.773111        4.050607     1.802588           19.885299

입력값 비교 예측
{
  "input": {
    "height": 170.0,
    "weight": 65.0
  },
  "predictions": {
    "nori": {
      "chest": 95.2,
      "waist": 80.3,
      "hip": 93.4,
      "thigh": 55.3,
      "calf": 36.6,
      "arm": 31.2,
      "shoulder": 38.5
    }
  }
}

상세 지표: /content/drive/MyDrive/Colab Notebooks/colab_nori/metrics.json


,model,target,mae,rmse,r2,p90_absolute_error,fit_seconds,predict_ms_per_row
0,nori,chest,2.502219,3.186318,0.865420,4.981266,1.802588,19.885299
1,nori,waist,3.332211,4.150272,0.832823,6.479551,1.802588,19.885299
2,nori,hip,2.385558,3.073804,0.743767,5.129381,1.802588,19.885299
3,nori,thigh,2.278407,2.930113,0.645517,4.855470,1.802588,19.885299
4,nori,calf,1.121121,1.451097,0.775911,2.347833,1.802588,19.885299
5,nori,arm,1.085295,1.437569,0.818439,2.150600,1.802588,19.885299
6,nori,shoulder,1.154987,1.463793,0.729902,2.410149,1.802588,19.885299


### 2-3. AutoGluon TabPFNMix Regressor

이 모델은 회귀를 위한 사전 학습된 AutoGluon 예측기입니다. AutoGluon의 `TabularPredictor`는 Hugging Face에 저장된 모델을 직접 로드할 수 있습니다.

In [43]:
import json
from pathlib import Path
import pandas as pd

MODEL = "tabpfn_mix"
artifact_dir = Path("/content/drive/MyDrive/Colab Notebooks") / f"colab_{MODEL}"

!python "/content/drive/MyDrive/Colab Notebooks/benchmark.py" benchmark \
    --data "/content/drive/MyDrive/Colab Notebooks/sizekorea_measurements_clean.csv" \
    --models {MODEL} \
    --artifact-dir "{artifact_dir}" \
    --height 170 \
    --weight 65

metrics_path = artifact_dir / "metrics.json"
if metrics_path.exists():
    df = pd.DataFrame(json.loads(metrics_path.read_text(encoding="utf-8")))
    display(df)
else:
    print(f"{MODEL}: metrics.json이 생성되지 않았습니다 — 위 실행 로그의 에러를 확인하세요.")

     model  mean_mae  mean_rmse  mean_r2  mean_p90_error  fit_seconds  predict_ms_per_row
tabpfn_mix  2.008125   2.561129 0.767588        4.094067    261.19747            3.469721

입력값 비교 예측
{
  "input": {
    "height": 170.0,
    "weight": 65.0
  },
  "predictions": {
    "tabpfn_mix": {
      "chest": 94.7,
      "waist": 80.8,
      "hip": 93.3,
      "thigh": 55.6,
      "calf": 36.5,
      "arm": 31.1,
      "shoulder": 38.5
    }
  }
}

상세 지표: /content/drive/MyDrive/Colab Notebooks/colab_tabpfn_mix/metrics.json


,model,target,mae,rmse,r2,p90_absolute_error,fit_seconds,predict_ms_per_row
0,tabpfn_mix,chest,2.546508,3.238158,0.861005,5.200912,261.19747,3.469721
1,tabpfn_mix,waist,3.367777,4.202849,0.828560,6.599998,261.19747,3.469721
2,tabpfn_mix,hip,2.434030,3.140833,0.732470,5.100002,261.19747,3.469721
3,tabpfn_mix,thigh,2.301391,2.948656,0.641016,4.757554,261.19747,3.469721
4,tabpfn_mix,calf,1.134439,1.462404,0.772405,2.400001,261.19747,3.469721
5,tabpfn_mix,arm,1.099175,1.452997,0.814521,2.200000,261.19747,3.469721
6,tabpfn_mix,shoulder,1.173557,1.482008,0.723138,2.400002,261.19747,3.469721


## 3. 모델 비교

모델을 효과적으로 비교하려면 공통 테스트 데이터셋(특징 및 실제 레이블)이 필요합니다. 여기서는 더미 데이터를 사용하여 RMSE(Root Mean Squared Error) 및 MAE(Mean Absolute Error)와 같은 메트릭을 계산하는 방법을 시연합니다.

In [44]:
import json
from pathlib import Path
import pandas as pd

MODELS = ["tabpfn_v2", "nori", "tabpfn_mix"]
base_dir = Path("/content/drive/MyDrive/Colab Notebooks")

detail_frames = []
for model in MODELS:
    metrics_path = base_dir / f"colab_{model}" / "metrics.json"
    if metrics_path.exists():
        detail_frames.append(pd.DataFrame(json.loads(metrics_path.read_text(encoding="utf-8"))))
    else:
        print(f"{model}: metrics.json이 없습니다 — 2번 섹션에서 먼저 실행하세요.")

if detail_frames:
    detail_df = pd.concat(detail_frames, ignore_index=True)
    summary_df = (
        detail_df.groupby("model", as_index=False)
        .agg(
            mean_mae=("mae", "mean"),
            mean_rmse=("rmse", "mean"),
            mean_r2=("r2", "mean"),
            mean_p90_error=("p90_absolute_error", "mean"),
            fit_seconds=("fit_seconds", "max"),
            predict_ms_per_row=("predict_ms_per_row", "max"),
        )
        .sort_values("mean_mae")
    )
    display(summary_df.round(4))
else:
    print("비교할 결과가 없습니다.")

,model,mean_mae,mean_rmse,mean_r2,mean_p90_error,fit_seconds,predict_ms_per_row
2,tabpfn_v2,1.9710,2.5145,0.7754,4.0302,8.2081,9.8496
0,nori,1.9800,2.5276,0.7731,4.0506,1.8026,19.8853
1,tabpfn_mix,2.0081,2.5611,0.7676,4.0941,261.1975,3.4697
